# 📈 S.O.M.I.S: Second-Order Market Impact Swarm
### Tactical Intelligence Launcher (Judge Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Avigyan-Das/SOMIS/blob/main/SOMIS_JUDGE_LAUNCHER.ipynb)

This notebook sets up the **S.O.M.I.S** multi-agent swarm on Google Colab hardware. 

**Instructions:**
1. Go to `Runtime` -> `Change runtime type` and ensure **GPU Acceleration (T4)** is selected.
2. Click **Runtime** -> **Run all**.
3. Wait for the `Localtunnel` URL to appear at the bottom. Click it to open the Tactical Terminal.

In [ ]:
# 1. Setup Environment
!git clone https://github.com/Avigyan-Das/SOMIS.git
%cd SOMIS
!pip install -r requirements.txt
!pip install pyngrok uvicorn httpx scrapling
!apt-get install -y libopenblas-dev
!npm install -g localtunnel

In [ ]:
# 2. Download Phi-3.5 Brain & KoboldCpp (Linux)
import os
os.makedirs('/content/SOMIS/models', exist_ok=True)
os.chdir('/content/SOMIS')

# Using a pinned, verified URL for the Linux CUDA binary
print("📥 Downloading KoboldCpp Engine...")
!wget -q --show-progress https://github.com/LostRuins/koboldcpp/releases/download/v1.76/koboldcpp-linux-x64-cuda1210 -O koboldcpp
!chmod +x koboldcpp

# Verify binary integrity
if os.path.getsize('koboldcpp') < 1000000:
    print("❌ ERROR: Download failed or file corrupted.")
else:
    print("✅ Engine Ready.")

print("📥 Downloading Phi-3.5 Model...")
!wget -q --show-progress https://huggingface.co/bartowski/Phi-3.5-mini-instruct-GGUF/resolve/main/Phi-3.5-mini-instruct-Q4_K_M.gguf -O models/phi-3.5.gguf

In [ ]:
# 3. Launch System Components
import subprocess
import time
import os

print("🚀 Launching S.O.M.I.S - The Brain (KoboldCpp)...")
kobold_path = os.path.abspath("koboldcpp")
model_path = os.path.abspath("models/phi-3.5.gguf")

subprocess.Popen([kobold_path, "--model", model_path, "--port", "5001", "--api"], 
                 stdout=subprocess.DEVNULL)

print("⏳ Waiting 45s for Model to load into GPU...")
time.sleep(45)

print("📡 Launching S.O.M.I.S - Command Center (Backend)...")
subprocess.Popen(["python", "server.py"], stdout=subprocess.DEVNULL)

print("🎨 Launching S.O.M.I.S - Tactical UI (Frontend)...")
os.chdir("/content/SOMIS/frontend")
subprocess.Popen(["npm", "install"], stdout=subprocess.DEVNULL).wait()
subprocess.Popen(["npm", "run", "dev"], stdout=subprocess.DEVNULL)

print("⏳ Waiting 15s for Vite UI server to initialize...")
time.sleep(15) 

print("🌐 EXPOSING TERMINAL TO PUBLIC URL...")
!npx localtunnel --port 5173